# 05 - Comparación de variables entre dos fincas

Cajas o rangos sombreados por día/semana desde el inicio de la etapa, con bandas y guardado SVG/PDF.

> Requiere `ipympl` (`pip install ipympl`) para ver valores al hacer clic. Usa `config.py` y `funciones_comunes.py` (misma carpeta).

In [1]:
# =============================================================================
# LIBRERÍAS
# =============================================================================
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
import ipywidgets as widgets
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
# Gradientes de color sobre filas sin datos válidos (sensor descartado): aviso inofensivo de pandas
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [2]:
%matplotlib widget

In [3]:
# =============================================================================
# CONFIGURACION Y FUNCIONES COMUNES (editar config.py y reiniciar el kernel)
# =============================================================================
from config import *
from funciones_comunes import (_limites_rango, obtener_rangos, periodo_finca, filtrar_etapa, columnas_analisis,
                               cargar_finca, cargar_bd_fija, cargar_cropx, cargar_bd_cropx,
                               mostrar_sin_datos, contar_por_banda, _ajustar_slider)

In [4]:
# Telemetría fija y CropX de todas las fincas, recortadas al periodo de sus etapas (incluye Descanso)
BD_FIJA_FILTERED = cargar_bd_fija()
BD_CROPX = cargar_bd_cropx()
BASES = {NOMBRE_BD_FIJA: BD_FIJA_FILTERED, NOMBRE_BD_CROPX: BD_CROPX}


Cargando datos de la finca: La_Orquidea
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para La_Orquidea: 17447 filas y 43 columnas.

Cargando datos de la finca: Venecia
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Venecia: 17199 filas y 43 columnas.

Cargando datos de la finca: Puerto_Arturo
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Puerto_Arturo: 17064 filas y 43 columnas.
La_Orquidea: fechas en reloj de 12 h sin AM/PM -> 8733 registros pasados a la tarde según el orden del archivo.
La_Orquidea: 17449 registros CropX, 2025-10-21 10:00:00 a 2026-10-19 22:30:00
Venecia: fechas en reloj de 12 h sin AM/PM -> 8736 registros pasados a la tarde según el orden del archivo.
Venecia: 17450 registros CropX, 2025-10-21 11:00:00 a 2026-10-19 23:30:00
Puerto_Arturo: fechas en reloj de 12 h sin AM/PM -> 3360 re

### Comparación variables entre fincas

In [5]:
# =============================================================================
# COMPARACION ENTRE DOS FINCAS: CAJAS O RANGOS SOMBREADOS POR DIA/SEMANA (figura en inglés)
# Para ver valores al hacer clic: pip install ipympl  y ejecutar  %matplotlib widget
# =============================================================================
import matplotlib
from matplotlib import cbook
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def etiqueta_en(variable: str) -> str:
    """Convierte el nombre de la columna en una etiqueta en inglés con unidades.

    Args:
        variable (str): Nombre de la columna (p. ej. "bottom_phosphorus_60cm").

    Returns:
        str: Etiqueta para el eje (p. ej. "Phosphorus at 60 cm (mg/kg)").
    """
    fijas = {"radiation": r"Solar radiation (W m$^{-2}$)",
             "air_temperature": r"Air temperature ($\degree$C)",
             "air_humidity": "Air relative humidity (%)"}
    if variable in fijas:
        return fijas[variable]
    cropx = re.fullmatch(r"(Moisture|Temperature|ECpw) (\d+)", variable)     # columnas CropX
    if cropx:
        nombres = {"Moisture": "Soil moisture", "Temperature": r"Soil temperature",
                   "ECpw": "Pore water EC"}
        unidades_cropx = {"Moisture": "%", "Temperature": r"$\degree$C", "ECpw": "dS/m"}
        tipo, profundidad = cropx.groups()
        return f"{nombres[tipo]} at {profundidad} cm ({unidades_cropx[tipo]}) - CropX"
    unidades = {"nitrogen": "Nitrogen (mg/kg)", "phosphorus": "Phosphorus (mg/kg)",
                "potassium": "Potassium (mg/kg)", "ph": "Soil pH",
                "moisture": "Soil moisture (%)", "temperature": r"Soil temperature ($\degree$C)",
                "conductivity": "Soil EC (dS/m)"}
    partes = variable.split("_")
    if len(partes) == 3 and partes[1] in unidades and partes[2].endswith("cm"):
        nombre, unidad = unidades[partes[1]].rsplit(" (", 1) if "(" in unidades[partes[1]] else (unidades[partes[1]], "")
        profundidad = partes[2].replace("cm", " cm")
        return f"{nombre} at {profundidad}" + (f" ({unidad}" if unidad else "")
    return variable


def inicio_etapa(finca: str, etapa: str) -> pd.Timestamp:
    """Fecha de inicio de la etapa (o del ciclo) de una finca.

    Args:
        finca (str): Nombre de la finca.
        etapa (str): Nombre de la etapa o "Ciclo completo".

    Returns:
        pd.Timestamp: Inicio del rango.
    """
    f = FINCAS_POR_NOMBRE[finca]
    inicio = f.fecha_inicio if etapa == "Ciclo completo" else obtener_rangos(f)[etapa][0]
    return pd.Timestamp(inicio)


def cajas_relativas(serie: pd.Series, t0: pd.Timestamp, dias: int) -> dict[int, dict]:
    """Estadísticas de boxplot por periodo contado desde el inicio de la etapa.

    Args:
        serie (pd.Series): Variable con índice temporal (ya filtrada a la etapa).
        t0 (pd.Timestamp): Inicio de la etapa (periodo 1).
        dias (int): Duración del periodo en días (1 = día, 7 = semana).

    Returns:
        dict[int, dict]: {periodo: estadísticas de `cbook.boxplot_stats` + n, desde, hasta}.
    """
    serie = serie.dropna()
    periodo = ((serie.index - t0).days // dias + 1).to_numpy()
    cajas = {}
    for k in np.unique(periodo):
        v = serie[periodo == k]
        st = cbook.boxplot_stats(v.to_numpy())[0]
        st.update(n=len(v), desde=v.index.min(), hasta=v.index.max())
        cajas[int(k)] = st
    return cajas


def dibujar_rangos(ax: plt.Axes, cajas_f: dict[int, dict], max_k: int, color: str) -> None:
    """Dibuja mediana (línea), Q1-Q3 (sombra oscura) y rango de bigotes (sombra clara) de una finca.

    Args:
        ax (plt.Axes): Eje donde dibujar.
        cajas_f (dict[int, dict]): Salida de `cajas_relativas` para una finca.
        max_k (int): Último periodo del eje x (los periodos sin datos cortan la sombra).
        color (str): Color de la finca.

    Returns:
        None
    """
    k = np.arange(1, max_k + 1)
    tabla = pd.DataFrame({p: {c: cajas_f[p][c] for c in ("med", "q1", "q3", "whislo", "whishi")}
                          for p in cajas_f}).T.reindex(k)
    ax.fill_between(k, tabla["whislo"], tabla["whishi"], color=color, alpha=0.12, linewidth=0)
    ax.fill_between(k, tabla["q1"], tabla["q3"], color=color, alpha=0.32, linewidth=0)
    ax.plot(k, tabla["med"], color=color, linewidth=1.2, marker="o", markersize=2.5)


def figura_comparacion(cajas: dict[str, dict[int, dict]], variable: str, nombre_periodo: str,
                       titulo: str | None, bandas: list[tuple[float, float]],
                       tipo: str = "Cajas") -> tuple[plt.Figure, plt.Axes, dict]:
    """Dibuja las dos fincas por periodo como boxplots lado a lado o como rangos sombreados superpuestos.

    Args:
        cajas (dict[str, dict[int, dict]]): {finca: salida de `cajas_relativas`}.
        variable (str): Columna graficada (para la etiqueta del eje y).
        nombre_periodo (str): "Day" o "Week".
        titulo (str | None): Título de la figura en inglés; None para no mostrarlo.
        bandas (list[tuple[float, float]]): Bandas activas (mínimo, máximo).
        tipo (str): "Cajas" (boxplots) o "Rangos" (sombras superpuestas en el mismo eje).

    Returns:
        tuple[plt.Figure, plt.Axes, dict]: Figura, eje y {(x, finca): estadísticas} para el clic.
    """
    desplazamiento = (-0.21, 0.21) if tipo == "Cajas" else (0.0, 0.0)
    max_k = max((max(c) for c in cajas.values() if c), default=1)
    por_x = {}

    with plt.rc_context(ESTILO_REVISTA), plt.ioff():
        fig, ax = plt.subplots(figsize=(7.16, 3.4), dpi=150, layout="constrained")   # doble columna
        for (finca, cajas_f), color, dx in zip(cajas.items(), COLORES_FINCAS, desplazamiento):
            if not cajas_f:
                continue
            pos = [k + dx for k in cajas_f]
            por_x.update({(round(p, 2), finca): st for p, st in zip(pos, cajas_f.values())})
            if tipo == "Rangos":
                dibujar_rangos(ax, cajas_f, max_k, color)
                continue
            ax.bxp(list(cajas_f.values()), positions=pos, widths=0.36, patch_artist=True,
                   boxprops={"facecolor": color, "alpha": 0.55, "edgecolor": color, "linewidth": 0.7},
                   medianprops={"color": "black", "linewidth": 0.9},
                   whiskerprops={"color": color, "linewidth": 0.7},
                   capprops={"color": color, "linewidth": 0.7},
                   flierprops={"marker": "o", "markersize": 1.5, "markerfacecolor": color,
                               "markeredgecolor": "none", "alpha": 0.5},
                   manage_ticks=False)

        manejadores = [Patch(facecolor=c, alpha=0.55, edgecolor=c, label=f.replace("_", " "))
                       for f, c in zip(cajas, COLORES_FINCAS)]
        if tipo == "Rangos":
            manejadores += [Line2D([], [], color="0.3", linewidth=1.2, marker="o", markersize=2.5, label="Median"),
                            Patch(facecolor="0.3", alpha=0.32, label="IQR (Q1-Q3)"),
                            Patch(facecolor="0.3", alpha=0.12, label="Whisker range (1.5 IQR)")]
        for (bmin, bmax), ls in zip(bandas, ESTILOS_BANDAS):
            ax.axhspan(bmin, bmax, color="0.5", alpha=0.08, zorder=0)
            for y in (bmin, bmax):
                ax.axhline(y, color="0.25", linestyle=ls, linewidth=0.8, zorder=1)
            manejadores.append(Line2D([], [], color="0.25", linestyle=ls, linewidth=0.8,
                                      label=f"Band {bmin:.2f} - {bmax:.2f}"))

        paso = next((p for p in (1, 2, 5, 7, 10, 14, 20, 30) if max_k / p <= 20), 30)
        ax.set_xlim(0.4, max_k + 0.6)
        ax.set_xticks(range(1, max_k + 1), minor=True)
        ax.set_xticks(sorted({1, *range(paso, max_k + 1, paso)}))
        ax.set_xlabel(f"{nombre_periodo} since stage start")
        ax.set_ylabel(etiqueta_en(variable))
        ax.grid(axis="y", color="0.9", linewidth=0.5)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        fig.legend(handles=manejadores, frameon=False, ncol=min(len(manejadores), 5 if tipo == "Cajas" else 4),
                   loc="outside lower center")
        if titulo:
            fig.suptitle(titulo, fontsize=9)
    return fig, ax, por_x


def activar_clic(fig: plt.Figure, ax: plt.Axes, por_x: dict, nombre_periodo: str,
                 tolerancia: float = 0.25) -> None:
    """Muestra las estadísticas del periodo más cercano al hacer clic (requiere %matplotlib widget).

    Args:
        fig (plt.Figure): Figura.
        ax (plt.Axes): Eje con los boxplots o rangos.
        por_x (dict): {(x, finca): estadísticas} devuelto por `figura_comparacion`.
        nombre_periodo (str): "Day" o "Week".
        tolerancia (float): Distancia máxima en x entre el clic y el periodo (0.25 cajas, 0.5 rangos).

    Returns:
        None
    """
    nota = ax.annotate("", xy=(0, 0), xytext=(10, 10), textcoords="offset points", fontsize=7.5,
                       fontfamily="serif",
                       bbox={"boxstyle": "round", "fc": "white", "ec": "0.4", "alpha": 0.95},
                       arrowprops={"arrowstyle": "-", "color": "0.4"}, visible=False, zorder=10)

    def _al_clic(evento):
        if evento.inaxes is not ax or evento.xdata is None:
            return
        candidatos = [(clave, st) for clave, st in por_x.items()
                      if abs(clave[0] - evento.xdata) <= tolerancia]
        if not candidatos:
            nota.set_visible(False)
        else:
            (x, finca), st = min(candidatos, key=lambda c: abs(c[1]["med"] - evento.ydata))
            k = round(x)
            nota.xy = (x, st["med"])
            nota.set_text(f"{finca.replace('_', ' ')} - {nombre_periodo} {k}\n"
                          f"{st['desde']:%d/%m/%y} to {st['hasta']:%d/%m/%y}\n"
                          f"Median: {st['med']:.2f}   Mean: {st['mean']:.2f}\n"
                          f"Q1-Q3: {st['q1']:.2f} - {st['q3']:.2f}   n = {st['n']}")
            nota.set_visible(True)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", _al_clic)


def panel_comparar_fincas(bd: dict[str, pd.DataFrame], nombre_bd: str, n_bandas: int = 3) -> None:
    """Widget para comparar una variable entre dos fincas por etapa, con bandas y guardado.

    Args:
        bd (dict[str, pd.DataFrame]): {finca: datos con índice temporal} (BD_FIJA_FILTERED o BD_CROPX).
        nombre_bd (str): Nombre de la base (nombre del archivo guardado).
        n_bandas (int): Número de sliders de bandas. Una banda con mínimo == máximo no se usa.

    Returns:
        None: Muestra los widgets, la figura y el conteo por banda de cada finca.
    """
    fincas = list(bd)
    if len(fincas) < 2:
        mostrar_sin_datos(f"Comparación de fincas - {nombre_bd}", "Se necesitan al menos dos fincas con datos")
        return
    w_a = widgets.Dropdown(options=fincas, value=fincas[0], description="Finca A")
    w_b = widgets.Dropdown(options=fincas, value=fincas[min(1, len(fincas) - 1)], description="Finca B")
    w_etapa = widgets.Dropdown(description="Etapa")
    w_var = widgets.Dropdown(description="Variable")
    w_agr = widgets.ToggleButtons(options=list(AGRUPACION), value="Semana", description="Agrupar")
    w_tipo = widgets.ToggleButtons(options=["Cajas", "Rangos"], value="Cajas", description="Gráfico")
    w_titulo = widgets.Checkbox(value=True, description="Mostrar título")
    w_bandas = [widgets.FloatRangeSlider(description=f"Banda {i + 1}", min=0, max=1, value=(0, 0),
                                         continuous_update=False, readout_format=".2f")
                for i in range(n_bandas)]
    w_archivo = widgets.Text(value=f"comparacion_fincas_{nombre_bd}", description="Archivo")
    w_formato = widgets.Dropdown(options=["svg", "pdf"], value="svg", description="Formato")
    w_guardar = widgets.Button(description="Guardar figura", icon="save")
    w_msj = widgets.Label()
    estado = {"fig": None}

    def _al_cambiar_variable(_=None):
        if w_var.value is None:
            return
        valores = pd.concat([bd[f][w_var.value] for f in (w_a.value, w_b.value)])
        for s in w_bandas:
            _ajustar_slider(s, float(valores.min()), float(valores.max()))

    def _al_cambiar_fincas(_=None):
        rangos_b = obtener_rangos(FINCAS_POR_NOMBRE[w_b.value])
        etapas = [e for e in obtener_rangos(FINCAS_POR_NOMBRE[w_a.value]) if e in rangos_b]
        etapa_anterior = w_etapa.value
        w_etapa.options = ["Ciclo completo", *etapas]
        w_etapa.value = etapa_anterior if etapa_anterior in w_etapa.options else "Ciclo completo"
        comunes = [c for c in columnas_analisis(bd[w_a.value])
                   if c in columnas_analisis(bd[w_b.value])]
        anterior = w_var.value
        w_var.options = comunes
        w_var.value = anterior if anterior in comunes else (comunes[0] if comunes else None)
        _al_cambiar_variable()

    def _guardar(_):
        if estado["fig"] is None:
            return
        DIR_SALIDA.mkdir(parents=True, exist_ok=True)
        ruta = DIR_SALIDA / f"{w_archivo.value or 'comparacion_fincas'}.{w_formato.value}"
        estado["fig"].savefig(ruta, format=w_formato.value, dpi=300, bbox_inches="tight")
        w_msj.value = f"Guardado: {ruta}"

    def _actualizar(finca_a, finca_b, etapa, variable, agrupacion, tipo, titulo, **bandas):
        if not variable or etapa is None:
            return
        if finca_a == finca_b:
            print("Selecciona dos fincas distintas.")
            return
        nombre_periodo, dias = AGRUPACION[agrupacion]
        cajas = {}
        for f in (finca_a, finca_b):
            df = filtrar_etapa(bd[f], f, None if etapa == "Ciclo completo" else etapa)
            cajas[f] = cajas_relativas(df[variable], inicio_etapa(f, etapa), dias)
        activas = [b for b in bandas.values() if b[0] != b[1]]
        texto = (f"{etiqueta_en(variable)} - {ETAPAS_EN.get(etapa, etapa)}: "
                 f"{finca_a.replace('_', ' ')} vs {finca_b.replace('_', ' ')}") if titulo else None
        if estado["fig"] is not None:
            plt.close(estado["fig"])
        fig, ax, por_x = figura_comparacion(cajas, variable, nombre_periodo, texto, activas, tipo)
        if not any(cajas.values()):                         # ninguna finca tiene datos: figura con aviso
            ax.text(0.5, 0.5, "No data in this stage", ha="center", va="center",
                    transform=ax.transAxes, color="0.4")
        estado["fig"] = fig

        if "ipympl" in matplotlib.get_backend().lower() or "widget" in matplotlib.get_backend().lower():
            fig.canvas.header_visible = False
            activar_clic(fig, ax, por_x, nombre_periodo, 0.25 if tipo == "Cajas" else 0.5)
            display(fig.canvas)
        else:
            display(fig)
            print("Para ver valores al hacer clic: pip install ipympl y ejecuta %matplotlib widget")

        if activas:
            tablas = {f: contar_por_banda(
                filtrar_etapa(bd[f], f, None if etapa == "Ciclo completo" else etapa)[variable],
                activas) for f in (finca_a, finca_b)}
            display(pd.concat(tablas, names=["Finca"]).style.format(
                {"% Dentro": "{:.2f} %", "% Fuera": "{:.2f} %"}))

    _al_cambiar_fincas()
    w_a.observe(_al_cambiar_fincas, "value")
    w_b.observe(_al_cambiar_fincas, "value")
    w_var.observe(_al_cambiar_variable, "value")
    w_guardar.on_click(_guardar)

    controles = {"finca_a": w_a, "finca_b": w_b, "etapa": w_etapa, "variable": w_var,
                 "agrupacion": w_agr, "tipo": w_tipo, "titulo": w_titulo,
                 **{f"b{i}": s for i, s in enumerate(w_bandas)}}
    salida = widgets.interactive_output(_actualizar, controles)
    display(widgets.VBox([
        widgets.HBox([widgets.VBox([w_a, w_b, w_etapa]), widgets.VBox([w_var, w_agr, w_tipo, w_titulo]),
                      widgets.VBox(w_bandas)]),
        widgets.HBox([w_archivo, w_formato, w_guardar, w_msj]),
        salida]))

In [6]:
panel_comparar_fincas(BD_FIJA_FILTERED, NOMBRE_BD_FIJA)

In [7]:
panel_comparar_fincas(BD_CROPX, NOMBRE_BD_CROPX)

In [8]:
# =============================================================================
# SERIES DE TIEMPO DE VARIAS FINCAS EN FECHAS REALES (interactivo, figura en inglés)
# =============================================================================
COLORES_SERIES = ("#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000")  # Okabe-Ito
RESOLUCION_SERIES = {"30 min": "30min", "Hora": "1h", "Día": "1D", "Semana": "W"}
RESOLUCION_EN = {"30 min": "30-min", "Hora": "Hourly", "Día": "Daily", "Semana": "Weekly"}
PERIODOS_EXTRA = ["Todo el periodo", "Ciclo completo"]


def color_finca(finca: str) -> str:
    """Color fijo de cada finca según su orden en config.FINCAS.

    Args:
        finca (str): Nombre de la finca.

    Returns:
        str: Color en hexadecimal.
    """
    nombres = list(FINCAS_POR_NOMBRE)
    return COLORES_SERIES[nombres.index(finca) % len(COLORES_SERIES)] if finca in nombres else "0.3"


def recortar_periodo(df: pd.DataFrame, finca: str, etapa: str) -> pd.DataFrame | None:
    """Recorta los datos de una finca al periodo elegido.

    Args:
        df (pd.DataFrame): Datos de la finca con índice temporal.
        finca (str): Nombre de la finca.
        etapa (str): "Todo el periodo", "Ciclo completo" o el nombre de una etapa.

    Returns:
        pd.DataFrame | None: Datos del periodo; None si la finca no tiene esa etapa.
    """
    if etapa == "Todo el periodo":
        return df
    if etapa == "Ciclo completo":
        return filtrar_etapa(df, finca, None)
    if etapa not in obtener_rangos(FINCAS_POR_NOMBRE[finca]):
        return None
    return filtrar_etapa(df, finca, etapa)


def dibujar_limites_etapas(ax: plt.Axes, finca: str, desde: pd.Timestamp, hasta: pd.Timestamp,
                           nivel: int = 0) -> None:
    """Líneas punteadas en el inicio de cada etapa de la finca, con el nombre en inglés.

    Args:
        ax (plt.Axes): Eje donde dibujar.
        finca (str): Nombre de la finca (define el color).
        desde (pd.Timestamp): Primera fecha graficada.
        hasta (pd.Timestamp): Última fecha graficada.
        nivel (int): Posición de la finca en la gráfica; baja los nombres para que no se monten entre fincas.

    Returns:
        None
    """
    color = color_finca(finca)
    separacion = (hasta - desde) * 0.04                   # etapas muy cortas: línea sin nombre
    ultimo = None
    for etapa, (inicio, fin) in obtener_rangos(FINCAS_POR_NOMBRE[finca]).items():
        t_ini = _limites_rango(inicio, fin)[0]
        if not desde <= t_ini <= hasta:
            continue
        ax.axvline(t_ini, color=color, linestyle=":", linewidth=0.8, zorder=1)
        if ultimo is None or t_ini - ultimo >= separacion:
            ax.text(t_ini, 0.98 - 0.32 * (nivel % 3), f" {ETAPAS_EN.get(etapa, etapa)}", color=color,
                    fontsize=6, rotation=90, va="top", ha="left", transform=ax.get_xaxis_transform())
            ultimo = t_ini


def figura_series_fincas(bd: dict[str, pd.DataFrame], fincas: list[str], variable: str, etapa: str,
                         resolucion: str, rango: bool, marcar: bool,
                         titulo: str | None) -> tuple[plt.Figure, plt.Axes, dict, list[str]]:
    """Serie de tiempo (media por periodo) de una variable en varias fincas, sobre fechas reales.

    Args:
        bd (dict[str, pd.DataFrame]): {finca: datos con índice temporal}.
        fincas (list[str]): Fincas a graficar.
        variable (str): Columna a graficar.
        etapa (str): "Todo el periodo", "Ciclo completo" o una etapa (cada finca usa sus propias fechas).
        resolucion (str): Clave de RESOLUCION_SERIES ("30 min", "Hora", "Día", "Semana").
        rango (bool): True para sombrear el mínimo-máximo de cada periodo.
        marcar (bool): True para marcar el inicio de cada etapa.
        titulo (str | None): Título en inglés; None para no mostrarlo.

    Returns:
        tuple[plt.Figure, plt.Axes, dict, list[str]]: Figura, eje, {finca: media/mín/máx por periodo}
            para el clic y avisos de fincas sin datos.
    """
    series, avisos = {}, []
    with plt.rc_context(ESTILO_REVISTA), plt.ioff():
        fig, ax = plt.subplots(figsize=(7.16, 3.4), dpi=150, layout="constrained")
        for finca in fincas:
            datos = recortar_periodo(bd[finca], finca, etapa)
            if datos is None:
                avisos.append(f"{finca}: no tiene la etapa '{etapa}'")
                continue
            if variable not in datos.columns or datos[variable].dropna().empty:
                avisos.append(f"{finca}: sin datos de {variable} en '{etapa}'")
                continue

            agregado = datos[variable].resample(RESOLUCION_SERIES[resolucion]).agg(["mean", "min", "max"])
            color = color_finca(finca)
            ax.plot(agregado.index, agregado["mean"], color=color, linewidth=1.1, label=finca.replace("_", " "))
            if rango:
                ax.fill_between(agregado.index, agregado["min"], agregado["max"],
                                color=color, alpha=0.15, linewidth=0)
            if marcar:
                dibujar_limites_etapas(ax, finca, agregado.index.min(), agregado.index.max(), len(series))
            series[finca] = agregado

        if series:
            manejadores, _ = ax.get_legend_handles_labels()
            if rango:
                manejadores.append(Patch(facecolor="0.3", alpha=0.15, label="Min-max range"))
            fig.legend(handles=manejadores, frameon=False, ncol=min(len(manejadores), 5),
                       loc="outside lower center")
        else:
            ax.text(0.5, 0.5, "No data for the selected farms and stage", ha="center", va="center",
                    transform=ax.transAxes, color="0.4")

        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%y"))
        ax.tick_params(axis="x", rotation=30)
        ax.set_xlabel("Date")
        ax.set_ylabel(f"{etiqueta_en(variable)}\n{RESOLUCION_EN[resolucion]} mean")
        ax.grid(color="0.9", linewidth=0.5)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        if titulo:
            fig.suptitle(titulo, fontsize=9)
    return fig, ax, series, avisos


def activar_clic_series(fig: plt.Figure, ax: plt.Axes, series: dict, distancia_px: float = 25) -> None:
    """Al hacer clic muestra finca, fecha, media, mínimo y máximo del punto más cercano (requiere ipympl).

    Args:
        fig (plt.Figure): Figura.
        ax (plt.Axes): Eje con las series.
        series (dict): {finca: media/mín/máx por periodo} devuelto por `figura_series_fincas`.
        distancia_px (float): Distancia máxima en píxeles entre el clic y el punto.

    Returns:
        None
    """
    nota = ax.annotate("", xy=(0, 0), xytext=(10, 10), textcoords="offset points", fontsize=7.5,
                       fontfamily="serif",
                       bbox={"boxstyle": "round", "fc": "white", "ec": "0.4", "alpha": 0.95},
                       arrowprops={"arrowstyle": "-", "color": "0.4"}, visible=False, zorder=10)

    def al_clic(evento):
        if evento.inaxes is not ax or evento.xdata is None:
            return
        fecha = pd.Timestamp(mdates.num2date(evento.xdata)).tz_localize(None)
        mejor = None
        for finca, agregado in series.items():
            validos = agregado.dropna(subset=["mean"])
            if validos.empty:
                continue
            i = validos.index.get_indexer([fecha], method="nearest")[0]
            punto = (mdates.date2num(validos.index[i]), validos["mean"].iloc[i])
            x_px, y_px = ax.transData.transform(punto)
            distancia = np.hypot(x_px - evento.x, y_px - evento.y)
            if mejor is None or distancia < mejor[0]:
                mejor = (distancia, finca, validos.index[i], validos.iloc[i], punto)

        if mejor is None or mejor[0] > distancia_px:
            nota.set_visible(False)
        else:
            _, finca, momento, fila, punto = mejor
            nota.xy = punto
            nota.set_text(f"{finca.replace('_', ' ')}\n{momento:%d/%m/%Y %H:%M}\n"
                          f"Mean: {fila['mean']:.2f}\nMin-max: {fila['min']:.2f} - {fila['max']:.2f}")
            nota.set_visible(True)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", al_clic)


def panel_series_fincas(bases: dict[str, dict[str, pd.DataFrame]]) -> None:
    """Widget: base, fincas (una o varias), variable, periodo, resolución, rango, etapas y guardado.

    Args:
        bases (dict[str, dict[str, pd.DataFrame]]): {nombre de la base: {finca: datos}}, p. ej. BASES.

    Returns:
        None: Muestra los widgets y la figura.
    """
    bases = {nombre: bd for nombre, bd in bases.items() if bd}
    if not bases:
        mostrar_sin_datos("Series de tiempo", "No hay bases con datos cargados")
        return

    w_base = widgets.Dropdown(options=list(bases), description="Base")
    w_fincas = widgets.SelectMultiple(description="Fincas", rows=4)
    w_var = widgets.Dropdown(description="Variable")
    w_etapa = widgets.Dropdown(description="Etapa")
    w_res = widgets.ToggleButtons(options=list(RESOLUCION_SERIES), value="Día", description="Resolución")
    w_rango = widgets.Checkbox(value=True, description="Rango mín-máx")
    w_marcar = widgets.Checkbox(value=True, description="Marcar etapas")
    w_titulo = widgets.Checkbox(value=True, description="Mostrar título")
    w_archivo = widgets.Text(value="series_fincas", description="Archivo")
    w_formato = widgets.Dropdown(options=["svg", "pdf"], value="svg", description="Formato")
    w_guardar = widgets.Button(description="Guardar figura", icon="save")
    w_msj = widgets.Label()
    estado = {"fig": None}

    def al_cambiar_fincas(cambio=None):
        bd = bases[w_base.value]
        fincas = list(w_fincas.value)
        variables = list(dict.fromkeys(c for f in fincas for c in columnas_analisis(bd[f])))
        etapas = list(dict.fromkeys(e for f in fincas for e in obtener_rangos(FINCAS_POR_NOMBRE[f])))
        var_anterior, etapa_anterior = w_var.value, w_etapa.value
        w_var.options = variables
        w_var.value = var_anterior if var_anterior in variables else (variables[0] if variables else None)
        w_etapa.options = [*PERIODOS_EXTRA, *etapas]
        w_etapa.value = etapa_anterior if etapa_anterior in w_etapa.options else PERIODOS_EXTRA[0]

    def al_cambiar_base(cambio=None):
        w_fincas.options = list(bases[w_base.value])
        w_fincas.value = tuple(w_fincas.options)               # todas seleccionadas al inicio
        al_cambiar_fincas()

    def guardar(boton):
        if estado["fig"] is None:
            return
        DIR_SALIDA.mkdir(parents=True, exist_ok=True)
        ruta = DIR_SALIDA / f"{w_archivo.value or 'series_fincas'}_{w_base.value}.{w_formato.value}"
        estado["fig"].savefig(ruta, format=w_formato.value, dpi=300, bbox_inches="tight")
        w_msj.value = f"Guardado: {ruta}"

    def actualizar(base, fincas, variable, etapa, resolucion, rango, marcar, titulo):
        if not fincas:
            print("Selecciona al menos una finca (Ctrl+clic para varias).")
            return
        if not variable or etapa is None:
            return
        texto = (f"{etiqueta_en(variable)} - {ETAPAS_EN.get(etapa, etapa)}: "
                 + ", ".join(f.replace("_", " ") for f in fincas)) if titulo else None
        if estado["fig"] is not None:
            plt.close(estado["fig"])
        fig, ax, series, avisos = figura_series_fincas(bases[base], list(fincas), variable, etapa,
                                                       resolucion, rango, marcar, texto)
        estado["fig"] = fig

        if "ipympl" in matplotlib.get_backend().lower() or "widget" in matplotlib.get_backend().lower():
            fig.canvas.header_visible = False
            activar_clic_series(fig, ax, series)
            display(fig.canvas)
        else:
            display(fig)
        for aviso in avisos:
            print(aviso)

    al_cambiar_base()
    w_base.observe(al_cambiar_base, "value")
    w_fincas.observe(al_cambiar_fincas, "value")
    w_guardar.on_click(guardar)

    salida = widgets.interactive_output(actualizar, {
        "base": w_base, "fincas": w_fincas, "variable": w_var, "etapa": w_etapa, "resolucion": w_res,
        "rango": w_rango, "marcar": w_marcar, "titulo": w_titulo})
    display(widgets.VBox([
        widgets.HBox([widgets.VBox([w_base, w_fincas]), widgets.VBox([w_var, w_etapa, w_res]),
                      widgets.VBox([w_rango, w_marcar, w_titulo])]),
        widgets.HBox([w_archivo, w_formato, w_guardar, w_msj]),
        salida]))

In [9]:
panel_series_fincas(BASES)